# CGM Forecasting Experiments — Continuation

This notebook documents the progression from the original CGM forecasting study to the final multimodal hybrid forecasting workflow. It is intended as a continuation of the earlier historical notebook, while keeping the original analysis intact.

The focus is on personalized blood glucose forecasting, multimodal feature integration, multi-horizon prediction, and the final methodological pipeline that was validated in the repository.

## 1. Project scope

The project is concerned with short-term glucose forecasting in Type 1 diabetes under a personalized, multi-horizon setting. The objective is not only to compare architectures, but to develop a forecasting framework that combines clinically relevant inputs with disease-specific temporal structure.

The final workflow includes:
- CGM dynamics
- Meal and carbohydrate information
- Insulin exposure
- Physiological signals when available
- 2020 wearable signal aggregation where the XML provides acceleration
- Multi-horizon targets at 15, 30, 60, 90, and 120 minutes
- Patient-wise evaluation and clinical metrics

In [ ]:
from pathlib import Path
import pandas as pd

repo = Path.cwd()
print(repo)
print('Data folder exists:', (repo / 'data').exists())
print('Reports folder exists:', (repo / 'reports').exists())

## 2. Data sources and cohorts

The repository uses the OhioT1DM dataset with separate patient cohorts and a multimodal preprocessing pipeline.

The two main cohort groups are:
- 2018 cohort: CGM + carbs + insulin + heart rate + steps
- 2020 cohort: CGM + carbs + insulin + acceleration

The common-feature baseline uses the shared inputs across both cohorts:
- glucose
- carbohydrate intake
- insulin exposure

In [ ]:
from pathlib import Path

for folder in ['data', 'data/phase2', 'output', 'reports']:
    p = Path(folder)
    print(f'{folder}:', 'present' if p.exists() else 'missing')

## 3. Feature engineering and acceleration audit

For the 2020 cohort, the wearable information is not discarded simply because heart rate and steps are unavailable. The XML data include acceleration information, and the preprocessing pipeline converts it to a 5-minute CGM-aligned representation.

The key idea is to aggregate acceleration using a strictly causal time window relative to the CGM timestamp. This preserves temporal validity and avoids leakage from future values.

Representative aggregated fields include:
- accel_mean_5m
- accel_std_5m
- accel_max_5m
- accel_min_5m
- accel_count_5m
- accel_coverage_5m
- accel_missing_5m

In [ ]:
import pandas as pd
from pathlib import Path

phase2_train = Path('data/phase2/ohio2020/train')
patient_files = sorted(phase2_train.glob('*.csv'))
print('Patient files:', [p.name for p in patient_files[:3]])

for fp in patient_files[:1]:
    df = pd.read_csv(fp)
    accel_cols = [c for c in df.columns if 'accel' in c.lower()]
    print(fp.name, 'accel columns:', accel_cols[:10])

## 4. Sequence construction and leakage control

The sequence pipeline enforces strict 5-minute indexing and prevents temporal leakage across train and test partitions. The aim is to ensure that all multimodal inputs are aligned to the CGM grid and that target generation uses a consistent forward horizon.

Important checks include:
- no future observations included in the input window
- strict 5-minute interval validation
- training-only scaling
- fixed random seeds for reproducibility
- correct target alignment for 15, 30, 60, 90, and 120 minutes

In [ ]:
HORIZONS = [15, 30, 60, 90, 120]
LOOKBACKS = [60, 120, 180, 240]
print('Forecast horizons:', HORIZONS)
print('Lookback windows:', LOOKBACKS)

## 5. Hybrid model design

The final architecture is a hybrid temporal model that combines complementary mechanisms rather than only stacking comparable blocks. The design uses:
- TCN branch for local and short-range temporal patterns
- GRU branch for sequential physiological dynamics
- Transformer branch for longer-range dependencies and cross-time interactions
- adaptive fusion layer to learn patient- and time-specific branch weighting
- multi-horizon prediction head

This structure is implemented in the repository as `HybridTCNGRUTransformer` and is designed to generate all five horizons from a shared representation.

In [ ]:
import inspect
from hybrid_models import HybridTCNGRUTransformer

print(HybridTCNGRUTransformer.__name__)
print(inspect.signature(HybridTCNGRUTransformer.__init__))

## 6. Baselines, ablations, and comparison structure

The final comparison includes the main architecture classes together with hybrid variants. The purpose of the ablation step is to determine whether each component contributes meaningfully to the final forecast quality.

The repo retains the base model families:
- LSTM
- GRU
- BiLSTM
- TCN
- Transformer

and evaluates hybrid combinations including:
- TCN-GRU
- GRU-Transformer
- TCN-Transformer
- TCN-GRU-Transformer
- adaptive hybrid model

## 7. Clinical and statistical evaluation

For each patient and horizon, the project reports clinically relevant metrics, including:
- RMSE
- MAE
- MARD
- Time Lag
- Clarke Error Grid Analysis

The patient-wise results are then summarized as mean ± SD across the cohort. Statistical comparisons are performed at the patient level, with care taken to use paired testing and correction for multiple comparisons where relevant.

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'horizon_min': [15, 30, 60, 90, 120],
    'final_model_focus': ['short-term', 'short-term', 'mid-term', 'mid-term', 'long-term']
})
print(summary)

## 8. Key findings and final interpretation

The evidence in the repository supports a nuanced conclusion rather than a blanket claim that one architecture is always best. The strongest interpretation is that performance depends systematically on the forecast horizon and personalization context.

In the final implementation, the hybrid architecture remains the most structured and defensible proposal because it combines complementary temporal mechanisms and supports multimodal forecasting with patient-wise evaluation.

## 9. Final status

This continuation notebook documents the final project state, while preserving the original historical workflow in the earlier notebook. The project is now presented as a completed multimodal blood glucose forecasting workflow with a validated hybrid architecture, clinically relevant evaluation, and evidence-backed research interpretation.